# Agentic SQL Assistant: Improving Queries with Reflection

This notebook demonstrates how a Large Language Model (LLM) can generate SQL queries and improve them through reflection and feedback.

**Experiment:** Identify the product colour with the highest total sales and compare two approaches:
1. Reflection using only the generated SQL query.
2. Reflection using both the SQL query and its execution results.

**Tools:** Python, OpenAI (via `aisuite`) and SQLite.


### Architecture

<p align="center">
  <img src="images/sql_reflection_workflow.png" width="425">
</p>

## 1. Setup

Import the required libraries and load environment variables, including the OpenAI API key, from the local `.env` file. Ensure `.env` is excluded from Git using `.gitignore`.


In [1]:
import json
import pandas as pd
import utils

from dotenv import load_dotenv

load_dotenv()

True

### 1.1 Initialise the LLM Client

`aisuite` provides a unified interface for interacting with different LLM providers. This notebook uses OpenAI models.


In [2]:
import aisuite as ai

client = ai.Client()

### 1.2 Prepare the SQLite Database

Create a local SQLite database (`products.db`) containing a `transactions` table with simulated product events, including sales, restocking and price updates.

**Note:** Rerunning the next cell recreates the `transactions` table and resets its data. No database server is required.


In [3]:
utils.create_transactions_db()

SQLite database 'products.db' created with a single 'transactions' table (event-sourced).


Inspect the table's columns and data types.

In [4]:
utils.print_html(utils.get_schema("products.db"))

### Understanding the Data

Each row represents a product event rather than a product summary.

- `action`: Event type (`insert`, `restock`, `sale` or `price_update`).
- `qty_delta`: Change in stock quantity (positive, negative or zero).
- `unit_price`: Price recorded for the event, where applicable.
- `color`, `brand`, `category`: Product attributes.

The SQL agent must interpret these fields correctly to calculate meaningful results.


## 2. Generate and execute SQL

### 2.1 Generate SQL from a Question

`generate_sql()` sends the question and database schema to an LLM and returns a SQL query. Defining the function does not make an API call.


In [5]:
def generate_sql(question: str, schema: str, model: str) -> str:
    prompt = f"""
    You are a SQL assistant. Generate a SQLite query that answers the user's question.

    Database schema:
    {schema}

    User question:
    {question}

    Return only the SQL query.
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    return response.choices[0].message.content.strip()

Generate a SQL query to identify the product colour with the highest total sales. This initial query is referred to as **V1**.


In [6]:
# Retrieve the database schema
schema = utils.get_schema("products.db")

# Define the user question
question = "Which color of product has the highest total sales?"

# Generate SQL (V1)
sql_V1 = generate_sql(question, schema, model="openai:gpt-4.1")

# Display the question and generated SQL
utils.print_html(question, title="User Question")
utils.print_html(sql_V1, title="SQL Query V1")


### 2.2 Execute V1

Execute the generated SQL query against the local SQLite database. A query may execute successfully but still produce an incorrect result due to logical errors.


In [7]:
# Execute SQL query V1
df_sql_V1 = utils.execute_sql(sql_V1, db_path="products.db")

# Display the results
utils.print_html(df_sql_V1, title="SQL Query V1 — Results")

color,total_sales
blue,-190571.46


**Observation:** A SQL query can execute successfully without producing the correct answer. We will investigate whether reflection can identify potential errors and improve the generated query.


## 3. Test SQL Reflection

Reflection involves using an LLM to evaluate and improve a generated SQL query. We compare two approaches:

1. **Reflection without execution feedback:** Evaluate the SQL query alone.
2. **Reflection with execution feedback:** Evaluate the SQL query alongside its execution results.


### 3.1 Reflection Without Execution Feedback

`refine_sql()` evaluates the original SQL query (V1) using the user question and database schema, without access to execution results.

It returns evaluation feedback and a refined query (V2).


In [8]:
def refine_sql(
    question: str,
    sql_query: str,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """Evaluate and refine SQL without execution feedback."""

    prompt = f"""
    You are a SQL reviewer. Evaluate the query and improve it if necessary.

    User question:
    {question}

    Database schema:
    {schema}

    Original SQL:
    {sql_query}

    Evaluate whether the SQL query correctly answers the question.
    If necessary, generate an improved SQLite query.
    Otherwise, return the original query unchanged.

    Return valid JSON with two fields:
    {{
        "feedback": "<brief explanation>",
        "refined_sql": "<SQL query>"
    }}
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    content = response.choices[0].message.content

    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip() or sql_query
    except (json.JSONDecodeError, TypeError):
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql


Run the reflection process and compare the original SQL query (V1) with the refined query (V2).

Does the LLM identify any logical errors or suggest meaningful improvements?

In [9]:
# Reflect on V1 without execution feedback
feedback, sql_V2 = refine_sql(
    question=question,
    sql_query=sql_V1,
    schema=schema,
    model="openai:gpt-4.1"
)

# Execute the refined query
df_sql_V2 = utils.execute_sql(sql_V2, db_path="products.db")

# Display the comparison
utils.print_html(sql_V1, title="Original SQL (V1)")
utils.print_html(df_sql_V1, title="Results of V1")

utils.print_html(feedback, title="Reflection Feedback")

utils.print_html(sql_V2, title="Refined SQL (V2)")
utils.print_html(df_sql_V2, title="Results of V2")

color,total_sales
blue,-190571.46


color,total_sales
blue,-190571.46


**Observation:** In our initial experiment, the LLM failed to identify the logical error and returned the original SQL query unchanged.

This demonstrates that reflection without execution feedback may fail to detect logical errors.

**Note:** Results may vary between runs and with different models or prompts.


### 3.2 Reflection With Execution Feedback

`refine_sql_external_feedback()` evaluates the original SQL query (V1) using the user question, database schema and actual execution results.

This allows the LLM to identify potential errors and refine the query based on external feedback.


In [10]:
def refine_sql_external_feedback(
    question: str,
    sql_query: str,
    df_feedback: pd.DataFrame,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """Evaluate and refine SQL using execution feedback."""

    prompt = f"""
    You are a SQL reviewer. Evaluate the query and improve it if necessary.

    User question:
    {question}

    Database schema:
    {schema}

    Original SQL:
    {sql_query}

    Execution results:
    {df_feedback.to_markdown(index=False)}

    Evaluate whether the SQL query and its results correctly answer the question.
    If necessary, generate an improved SQLite query.
    Otherwise, return the original query unchanged.

    Return valid JSON with two fields:
    {{
        "feedback": "<brief explanation>",
        "refined_sql": "<SQL query>"
    }}
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    content = response.choices[0].message.content

    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip() or sql_query
    except (json.JSONDecodeError, TypeError):
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql

Execute V1 and pass its results to the reflection function. Then execute the refined query (V2) to evaluate the improvement.

In [11]:
# Reflect on V1 using execution feedback
feedback, sql_V2 = refine_sql_external_feedback(
    question=question,
    sql_query=sql_V1,
    df_feedback=df_sql_V1,
    schema=schema,
    model="openai:gpt-4.1"
)

# Execute the refined query
df_sql_V2 = utils.execute_sql(sql_V2, db_path="products.db")

# Display the comparison
utils.print_html(sql_V1, title="Original SQL (V1)")
utils.print_html(df_sql_V1, title="Results of V1")

utils.print_html(feedback, title="Reflection Feedback")

utils.print_html(sql_V2, title="Refined SQL (V2)")
utils.print_html(df_sql_V2, title="Results of V2")


color,total_sales
blue,-190571.46


color,total_sales
white,358315.09


**Observation:** In our initial experiment, reflection with execution feedback successfully identified the incorrect sales calculation and refined the SQL query to use `SUM(ABS(qty_delta) * unit_price)`.

The refined query (V2) returned **white** as the product colour with the highest total sales (**358,315.09**).

**Note:** These results are specific to the initial experiment. LLM-generated queries and their results may vary between runs.


## 4. Combine the Steps into One Workflow

`run_sql_workflow()` combines SQL generation, execution and reflection into a single process:

1. Retrieve the database schema.
2. Generate the initial SQL query (V1).
3. Execute V1.
4. Evaluate V1 using its execution results and generate a refined query (V2).
5. Execute V2.

**Limitation:** The workflow performs only one reflection iteration. It does not automatically retry until the query is correct or independently validate the final answer.


In [12]:
def run_sql_workflow(
    db_path: str,
    question: str,
    model_generation: str = "openai:gpt-4.1",
    model_evaluation: str = "openai:gpt-4.1",
):
    """Generate, execute and refine SQL using execution feedback."""

    # Step 1: Retrieve database schema
    schema = utils.get_schema(db_path)
    utils.print_html(schema, title="1. Database Schema")

    # Step 2: Generate SQL (V1)
    sql_v1 = generate_sql(question, schema, model_generation)
    utils.print_html(sql_v1, title="2. Generated SQL (V1)")

    # Step 3: Execute V1
    df_v1 = utils.execute_sql(sql_v1, db_path)
    utils.print_html(df_v1, title="3. Results of V1")

    # Step 4: Reflect and refine
    feedback, sql_v2 = refine_sql_external_feedback(
        question=question,
        sql_query=sql_v1,
        df_feedback=df_v1,
        schema=schema,
        model=model_evaluation,
    )
    utils.print_html(feedback, title="4. Reflection Feedback")
    utils.print_html(sql_v2, title="4. Refined SQL (V2)")

    # Step 5: Execute V2
    df_v2 = utils.execute_sql(sql_v2, db_path)
    utils.print_html(df_v2, title="5. Results of V2")

### 4.1 Run the Complete Workflow

Run the complete workflow using `openai:gpt-4.1` for both SQL generation and reflection.

The workflow makes OpenAI API calls and displays the initial query (V1), its execution results, reflection feedback and the refined query (V2) with its results.


In [13]:
run_sql_workflow(
    db_path="products.db",
    question="Which color of product has the highest total sales?",
    model_generation="openai:gpt-4.1",
    model_evaluation="openai:gpt-4.1",
)

color,total_sales
blue,-190571.46


color,total_sales
white,358315.09


## 5. Key Findings and Future Improvements

This experiment demonstrates that:

- LLM-generated SQL can be syntactically correct but logically incorrect.
- Reflection without execution feedback failed to identify the error in our initial experiment.
- Reflection with execution feedback successfully identified and corrected the error.
- Successful SQL execution does not guarantee that the results are correct.

**Future improvements:**

- **SQL safety:** Restrict execution to read-only queries.
- **Error recovery:** Automatically detect and attempt to correct SQL execution errors.
- **Iterative reflection:** Allow multiple refinement attempts with a maximum retry limit.
- **Evaluation:** Test the agent against questions with independently verified answers.

These improvements will be implemented incrementally, keeping the baseline notebook simple and focused.